# Tensor BSPF in 2D and 3D

Install `python -m pip install -e '.[host,notebook,test]'` from the repository root and select that Python kernel.
Spatial axes follow **(x,y,z)**, with `meshgrid(indexing="ij")`. This differs
from image-style `(ny,nx)` ordering. Trailing dimensions may hold batches.

For $f(x,y)=x^2+y^2$, $\nabla f=(2x,2y)$ and $\Delta f=4$.


In [ ]:
import pybspf.calculus as bspf_calculus
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pybspf as bspf

x, y = jnp.linspace(-1, 1, 25), jnp.linspace(-1, 1, 27)
plan2 = bspf_plans.plan_2d(x, y, degree=3, n_basis=8, boundary_points=5)
X, Y = jnp.meshgrid(x, y, indexing="ij")
f2 = X**2 + Y**2
gradient = jax.jit(bspf_operators.gradient)(plan2, f2)
laplacian = jax.jit(bspf_operators.laplacian)(plan2, f2)
assert jnp.max(jnp.abs(gradient - jnp.stack([2*X, 2*Y]))) < 1e-8
assert jnp.max(jnp.abs(laplacian - 4)) < 1e-8
print("2D Laplacian error:", float(jnp.max(jnp.abs(laplacian - 4))))


## Extend by one axis, without another solver implementation

For $f(x,y,z)=x^2+y^2+z^2$, the Laplacian is 6 and its volume integral
on $[-1,1]^3$ is 8. The complete tensor split has eight sampled components.


In [ ]:
z = jnp.linspace(-1, 1, 29)
plan3 = bspf_plans.tensor_plan(*plan2.axes, bspf_plans.plan_1d(z, degree=3, n_basis=8, boundary_points=5))
X, Y, Z = jnp.meshgrid(x, y, z, indexing="ij")
f3 = X**2 + Y**2 + Z**2
laplacian3 = jax.jit(bspf_operators.laplacian)(plan3, f3)
components = jax.jit(bspf_operators.tensor_decompose)(plan3, f3)
volume_integral = jax.jit(bspf_calculus.integrate_box)(plan3, f3)
assert jnp.max(jnp.abs(laplacian3 - 6)) < 1e-8
assert jnp.max(jnp.abs(sum(components.values()) - f3)) < 1e-12
assert jnp.abs(volume_integral - 8) < 1e-10
print("components:", tuple(components), "integral:", float(volume_integral))


## Vector calculus

For a gradient field, curl should vanish to numerical roundoff. Components are
stored on a new leading axis: `(3,nx,ny,nz)`.


In [ ]:
vector = jax.jit(bspf_operators.gradient)(plan3, f3)
rotation = jax.jit(bspf_operators.curl)(plan3, vector)
assert jnp.max(jnp.abs(rotation)) < 1e-8
print("max curl of gradient:", float(jnp.max(jnp.abs(rotation))))
